# Nested Designs

A factor is **nested** when each of its levels belongs to exactly one level of another factor. Batches belong to suppliers; spindles belong to machines; repeated measurements belong to particular subjects. The grouping determines which comparisons and replications are meaningful.

## Nested Versus Crossed

In a crossed design, the same B levels occur at each A level: for example, the same temperatures are tested with every material. In a nested design, “batch 1” from supplier A is a different batch from “batch 1” from supplier B. Their shared label does not create a common factor level.

A separate batch main effect and supplier-by-batch interaction have no useful crossed interpretation in that setting. Model batch **within supplier**, written $B(A)$.

## Model & Notation

For $a$ A levels, $b$ B levels within each A, and $n$ observations per nested unit,

$$y_{ijk}=\mu+\tau_i+\beta_{j(i)}+\epsilon_{ijk}.$$

Here $i$ identifies the outer group, $j(i)$ identifies an inner unit belonging to that group, and $k$ identifies a replicate within that unit. In a fully fixed model, use $\sum_i\tau_i=0$ and $\sum_j\beta_{j(i)}=0$ separately for every $i$.

A bar means an average; a dot marks an averaged index. Thus $\bar y_{ij\cdot}$ is the nested-unit mean, $\bar y_{i\cdot\cdot}$ is the outer-group mean, and $\bar y_{\cdot\cdot\cdot}$ is the overall mean.

## Balanced ANOVA Decomposition

$$SS_A=bn\sum_i(\bar y_{i\cdot\cdot}-\bar y_{\cdot\cdot\cdot})^2,$$

$$SS_{B(A)}=n\sum_i\sum_j(\bar y_{ij\cdot}-\bar y_{i\cdot\cdot})^2,$$

$$SS_E=\sum_i\sum_j\sum_k(y_{ijk}-\bar y_{ij\cdot})^2.$$

| Source | df | Mean square |
|---|---:|---|
| A | $a-1$ | $SS_A/(a-1)$ |
| B within A | $a(b-1)$ | $SS_{B(A)}/[a(b-1)]$ |
| Error | $ab(n-1)$ | $SS_E/[ab(n-1)]$ |
| Total | $abn-1$ | — |

The three sums of squares add to $SS_T=\sum_{ijk}(y_{ijk}-\bar y_{\cdot\cdot\cdot})^2$.

:::{dropdown} Why the decomposition works
Split each centered observation into three pieces:

$$y_{ijk}-\bar y_{\cdot\cdot\cdot}=
(\bar y_{i\cdot\cdot}-\bar y_{\cdot\cdot\cdot})+
(\bar y_{ij\cdot}-\bar y_{i\cdot\cdot})+
(y_{ijk}-\bar y_{ij\cdot}).$$

Within-unit residuals sum to zero, and within-A deviations of unit means sum to zero. The cross-products therefore cancel when squared and summed, leaving the three ANOVA components.
:::

If arbitrary within-group labels are used to calculate a balanced crossed ANOVA, then algebraically

$$SS_{B(A)}=SS_B+SS_{AB},\qquad (b-1)+(a-1)(b-1)=a(b-1).$$

Only their combined nested quantity has the intended interpretation. Relabeling inner units within an outer group cannot change the nested sum of squares, although it can change the artificial B and AB pieces.

## Choose Tests from the Sampling Model

If both factors are fixed, test A and $B(A)$ against $MS_E$, under the usual independent normal equal-variance error assumptions.

If inner units are random, use independent $\beta_{j(i)}\sim N(0,\sigma_{B(A)}^2)$. Then

$$E[MS_E]=\sigma^2,\qquad
E[MS_{B(A)}]=\sigma^2+n\sigma_{B(A)}^2.$$

For fixed A,

$$E[MS_A]=\sigma^2+n\sigma_{B(A)}^2+\frac{bn}{a-1}\sum_i\tau_i^2.$$

For random A with variance $\sigma_A^2$,

$$E[MS_A]=\sigma^2+n\sigma_{B(A)}^2+bn\sigma_A^2.$$

In either of these **random-inner-unit** models, test A using $MS_A/MS_{B(A)}$, with $(a-1,a(b-1))$ df. Test the inner variance component using $MS_{B(A)}/MS_E$, with $(a(b-1),ab(n-1))$ df.

The reason is concrete: repeated readings from one batch do not provide new independent batches. Comparing suppliers must account for batch-to-batch variation, not just reading error. Do not apply the random-inner-unit rule solely because some factor is described as random; specify which factor is random and derive its expected mean squares.

## Worked Nested Calculation

The following small **illustrative dataset** has three machines, two distinct spindles within each machine, and four readings per spindle. It is constructed for practicing the decomposition.

In [1]:
import numpy as np
from scipy.stats import f
# Dimensions: machine, spindle within machine, repeated reading.
y = np.array([[[8,9,10,9], [12,13,14,13]],
              [[16,17,18,17], [12,13,14,13]],
              [[18,19,20,19], [22,23,24,23]]], dtype=float)
a,b,n = y.shape
grand = y.mean()
outer = y.mean(axis=(1,2))
inner = y.mean(axis=2)
SSA = b*n*np.sum((outer-grand)**2)
SSBA = n*np.sum((inner-outer[:,None])**2)
SSE = np.sum((y-inner[:,:,None])**2)
SST = np.sum((y-grand)**2)
assert np.isclose(SSA+SSBA+SSE, SST)
MSA, MSBA, MSE = SSA/(a-1), SSBA/(a*(b-1)), SSE/(a*b*(n-1))
for label, ss, df in [('A',SSA,a-1), ('B(A)',SSBA,a*(b-1)),
                      ('Error',SSE,a*b*(n-1))]:
    print(f'{label:6s} SS={ss:8.3f} df={df:2d} MS={ss/df:8.3f}')
print('A test with random spindles:', round(MSA/MSBA,4),
      'p =', round(f.sf(MSA/MSBA,a-1,a*(b-1)),6))
print('Spindle variance test:', round(MSBA/MSE,4),
      'p =', format(f.sf(MSBA/MSE,a*(b-1),a*b*(n-1)), ".3g"))

A      SS= 405.333 df= 2 MS= 202.667
B(A)   SS=  96.000 df= 3 MS=  32.000
Error  SS=  12.000 df=18 MS=   0.667
A test with random spindles: 6.3333 p = 0.083795
Spindle variance test: 48.0 p = 8.63e-09


The outer-group mean square is $202.6667$, the nested-unit mean square is 32, and the residual mean square is $2/3$. For random spindles, the machine test uses $202.6667/32=6.3333$, rather than the much larger $202.6667/(2/3)=304$. The appropriate denominator reflects which population the comparison is meant to cover.

### Combining an Existing Table

If a balanced table instead reports $SS_B=1.5$ on 1 df and $SS_{AB}=42.25$ on 2 df, the combined nested row is

$$SS_{B(A)}=43.75,\qquad df_{B(A)}=3,\qquad MS_{B(A)}=14.5833.$$

The error row stays unchanged. A reported ratio near 9.9 with $(3,18)$ df corresponds to an upper-tail p-value near 0.0004. Use the actual error mean square for an exact calculation rather than inferring it from a rounded F statistic.

## Variance Components & Replication

When both levels are random, moment estimates are

$$\widehat\sigma^2=MS_E,\quad
\widehat\sigma_{B(A)}^2=\frac{MS_{B(A)}-MS_E}{n},\quad
\widehat\sigma_A^2=\frac{MS_A-MS_{B(A)}}{bn}.$$

For the illustrative data these are $2/3$, $47/6\approx7.8333$, and $64/3\approx21.3333$, respectively. Negative moment estimates can arise in other datasets and need the same care as other variance-component estimates.

Two readings from the same inner unit share both random components; two readings from different inner units in the same outer group share only the A component. Distinct outer groups are independent under this model. Extra readings chiefly improve estimation of residual error; more independent inner and outer units are needed to learn their respective population variation.

## Deeper Nesting & Common Mistakes

For supplier, batch, sub-batch, and reading, a three-level model is

$$y_{ijkl}=\mu+u_i+v_{j(i)}+w_{k(ij)}+\epsilon_{ijkl}.$$

Each additional random grouping contributes its own variance component. Preserve unique unit identities in the data: “supplier A / batch 1” and “supplier B / batch 1” must not become the same batch.

These balanced formulas assume the same number of units and readings at each branch. Unequal branch sizes or missing observations call for a model-based analysis. Repeated instrument readings are not independent experimental replicates if they share an unmodeled source of variation. Define the unit of treatment assignment and the sampling hierarchy before fitting the model.